# 02 — Feature engineering, targets and chronological split

This notebook creates the same 15 features per pair and next-observation targets used in the final notebook. It saves each cleaned pair dataset so the modelling notebook can be run independently.

Run notebook 01 first to understand the source data. This notebook reloads the raw CSV and repeats the deterministic preparation steps.

In [2]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from fx_utils import (
    load_forex_data, build_pair_rates, build_features_and_targets,
    chronological_split, get_7_features, PAIRS, PROCESSED_DIR
)

forex_df = load_forex_data(PROJECT_ROOT / "data" / "daily_forex_rates.csv")
pair_df = build_pair_rates(forex_df)
features_df, targets_df, model_datasets = build_features_and_targets(pair_df)

print("Pair rates:", pair_df.shape)
print("All feature columns:", features_df.shape)
print("Target columns:", targets_df.shape)

ImportError: cannot import name 'build_features_and_targets' from 'fx_utils' (/Users/namithar/ML-Projects/FX-Forecasting/src/fx_utils.py)

In [ ]:
print("Feature count per pair:")
for pair in PAIRS:
    cols = [c for c in features_df.columns if c.startswith(f"{pair}_")]
    print(f"{pair}: {len(cols)} features")

print("\nFirst date with complete feature and target rows:")
for pair in PAIRS:
    data = model_datasets[pair]
    print(pair, data.shape, data.index.min(), "to", data.index.max(),
          "| missing values:", int(data.isna().sum().sum()))

In [ ]:
# Inspect the 7-feature set and the additional 8 features.
pair = "GBP_USD"
features_7 = get_7_features(pair)
features_15 = [c for c in features_df.columns if c.startswith(f"{pair}_")]
print("7-feature set:", features_7)
print("\nAdditional features:", [c for c in features_15 if c not in features_7])
display(model_datasets[pair].head())

In [ ]:
splits = {pair: chronological_split(model_datasets[pair]) for pair in PAIRS}
summary = []
for pair in PAIRS:
    for split_name, part in splits[pair].items():
        summary.append({
            "Currency Pair": pair,
            "Split": split_name,
            "Rows": len(part),
            "Start": part.index.min(),
            "End": part.index.max()
        })
display(pd.DataFrame(summary))

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
for pair in PAIRS:
    model_datasets[pair].to_csv(PROCESSED_DIR / f"{pair}_model_data.csv")
features_df.to_csv(PROCESSED_DIR / "all_features.csv")
pair_df.to_csv(PROCESSED_DIR / "pair_rates.csv")
print("Saved processed data to:", PROCESSED_DIR)

## Definitions to remember

- **7 features:** current log return, previous log return, and moving averages of returns over 5, 10, 20, 60 and 120 observations.
- **15 features:** the original 7 plus log exchange rate, its square and cube, and moving averages of log exchange rate over 5, 10, 20, 60 and 220 observations.
- **Target return:** log return from observation *t* to *t+1*.
- **Target direction:** 1 if that next return is positive, otherwise 0.
- **Split:** first 80% train, next 10% validation, final 10% test, without shuffling.